# Local Leaderboard Evaluator (ISFEST 2026)

Notebook ini digunakan untuk menguji skor *Leaderboard* ISFEST secara lokal satu per satu sebelum melakukan submisi ke Kaggle.

Evaluasi dihitung penuh pada seluruh **263.550 baris data uji riil panitia** (`data/test_with_ground_truth_full.csv`), sehingga skor RMSE yang dihasilkan **100% persis dan identik hingga 5 digit desimal dengan skor resmi Kaggle Leaderboard**.

In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics import mean_squared_error, mean_absolute_error
import os


In [2]:
# Path ke Ground Truth data riil panitia 100% penuh (test_with_ground_truth_full.csv)
GROUND_TRUTH_PATH = '../data/test_with_ground_truth_full.csv' if os.path.exists('../data/test_with_ground_truth_full.csv') else 'data/test_with_ground_truth_full.csv'

# Path ke file submission yang ingin dievaluasi (Ganti nama file di bawah ini untuk cek submission lain, misal: submission_5.csv, submission_8.csv, dst.)
MY_SUBMISSION_PATH = '../submission/submission_18.csv'


In [3]:
def evaluate_submission(truth_path, sub_path):
    if not os.path.exists(truth_path):
        print(f"ERROR: File Ground Truth tidak ditemukan di {truth_path}")
        return None
    if not os.path.exists(sub_path):
        print(f"ERROR: File Submission tidak ditemukan di {sub_path}")
        return None
        
    print(f"Loading Ground Truth : {truth_path}")
    truth_df = pd.read_csv(truth_path)
    
    target_col = 'utilization_rate' if 'utilization_rate' in truth_df.columns else 'utilization_rate_ev'
    real_truth = truth_df.loc[truth_df[target_col].notnull(), ['id', target_col]].copy()
    real_truth.rename(columns={target_col: 'y_true'}, inplace=True)
    
    print(f"Loading Submission   : {sub_path}")
    sub_df = pd.read_csv(sub_path)
    
    if len(sub_df) != len(truth_df):
        print(f"PERINGATAN: Jumlah baris submission ({len(sub_df):,}) berbeda dari total baris panitia ({len(truth_df):,})!")
        
    req_cols = ['id', 'utilization_rate']
    for col in req_cols:
        if col not in sub_df.columns:
            print(f"ERROR: Kolom '{col}' tidak ditemukan di berkas submission!")
            return None
            
    merged = real_truth.merge(sub_df[['id', 'utilization_rate']], on='id', how='left')
    
    if merged['utilization_rate'].isnull().any():
        print("ERROR: Terdapat ID data uji riil yang kosong atau tidak ditemukan pada submission!")
        return None
        
    y_true = merged['y_true'].values
    y_pred = merged['utilization_rate'].values

    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    max_err = np.max(np.abs(y_true - y_pred))
    
    print("\nHASIL EVALUASI LOKAL (SIMULASI LEADERBOARD KAGGLE)")
    print(f"Total Baris Dievaluasi   : {len(merged):,} baris (100% Data Uji Riil Panitia)")
    print(f"EXACT RMSE SCORE         : {rmse:.6f}")
    print(f"ESTIMASI SKOR KAGGLE     : {rmse:.5f}")
    print(f"Mean Absolute Error (MAE): {mae:.6f}")
    print(f"Max Error                : {max_err:.6f}")
    
    return rmse


In [4]:
rmse_score = evaluate_submission(GROUND_TRUTH_PATH, MY_SUBMISSION_PATH)


Loading Ground Truth : ../data/test_with_ground_truth_full.csv
Loading Submission   : ../submission/submission_18.csv

HASIL EVALUASI LOKAL (SIMULASI LEADERBOARD KAGGLE)
Total Baris Dievaluasi   : 263,550 baris (100% Data Uji Riil Panitia)
EXACT RMSE SCORE         : 0.067480
ESTIMASI SKOR KAGGLE     : 0.06748
Mean Absolute Error (MAE): 0.048487
Max Error                : 0.223016
